# Lab 2: CNN Architecture Design Decisions

**AutoParts Inc. — Manufacturing Intelligence Team**

**Context:** Building on Lab 1's discussion of feature hierarchies and receptive fields, this notebook works through the concrete architecture decisions for an automated visual inspection CNN: 
- kernel sizes, 
- pooling strategy, 
- network depth/width, and
- transfer learning 

all evaluated against the practical constraint of real-time inference on the production line, with 50,000 labeled images (defective vs. non-defective) available for training.

## 1. Kernel Size Decisions

I'd use small **3×3 kernels** through most of the network, stacked in sequence rather than relying on one large kernel.

- **Why 3×3, stacked:** Two stacked 3×3 convolutions produce the same 5×5 effective receptive field as a single 5×5 kernel, but with fewer parameters (2 × 9 = 18 weights per channel vs. 25) and an extra non-linearity in between. That extra non-linearity lets the network learn more complex, non-linear decision boundaries — useful for separating visually similar patterns like a hairline crack vs. an intentional machining groove — without paying for the larger kernel's parameter and compute cost.
- **The exception — the first layer:** A slightly larger kernel (5×5 or 7×7) at the input can help capture broader low-level structure (edges, gradients, coarse texture) across the varied geometries in the dataset — simple brackets vs. complex engine parts — before the network narrows its focus in later layers.
- **Trade-offs for mobile/edge deployment:** Kernel size directly drives both parameter count and FLOPs (compute scales roughly with `kernel_size² × in_channels × out_channels × output_spatial_size`). Since inspection needs to run in real time on the line, smaller kernels keep per-layer compute low, and stacking them preserves representational power — the standard trade-off VGG-style and mobile-oriented architectures (MobileNet, EfficientNet) are built around.

**Bottom line:** default to 3×3, use a larger kernel only at the input stem, and let depth (not kernel size) do the work of growing the receptive field.

## 2. Pooling Strategy

I'd use **max pooling (2×2, stride 2)** after groups of convolutional layers — not after every single layer — tapering pooling frequency in deeper stages.

- **Type — why max, not average:** Max pooling keeps the strongest activation in each window: the sharpest edge, the most pronounced texture irregularity. For defect detection, that's exactly the signal we care about — a crack or deformation is a *local extreme* in the feature map, and averaging would dilute it against surrounding normal texture.
- **Size and frequency:** 2×2/stride-2 pooling halves spatial resolution each time, which cuts computation substantially in every downstream layer — critical for real-time throughput on a production line. I'd avoid pooling after every conv layer, since defects (e.g., fine hairline cracks) can be spatially small, and over-aggressive downsampling in early layers risks losing them before deeper layers get a chance to combine that detail into a defect-level concept. Tapering pooling in later blocks preserves enough spatial resolution to localize where on the part a defect occurs, not just whether one is present.
- **Translation invariance:** Because pooling summarizes a local neighborhood into one value, small shifts in where a feature appears within that neighborhood don't change the output. This matters directly for our production-line setting: parts won't be perfectly centered or identically oriented as they move past the camera, and lighting/angle will vary slightly between units. Pooling makes the network robust to those minor positional shifts, so it recognizes "there's a crack near this rivet hole" regardless of a few pixels of part misalignment.

**Bottom line:** max pooling for feature-preserving downsampling, applied selectively (not every layer) to balance compute reduction against retaining enough spatial detail to localize small defects.

## 3. Network Depth and Width Considerations

Given the real-time, likely edge/mobile deployment target, I'd favor a **moderately deep but narrow** network over a very wide, shallow one, and lean on efficiency-oriented building blocks rather than brute-force scale.

- **Depth:** Depth is what builds the feature hierarchy from Lab 1 — edges → textures/shapes → whole-part, defect-specific concepts. Enough depth is needed to let the network integrate information across a large receptive field (to judge whether a deformation spans a meaningful portion of the part, not just a pixel). But each added layer adds latency, so depth should be added only as long as it measurably improves validation accuracy on defect classes — not by default.
- **Width (filters per layer):** More filters per layer let the network represent a greater variety of features at that stage (useful given the range of materials/textures across brackets vs. engine parts), but width scales compute and memory roughly linearly, and mobile devices are memory- and power-constrained. I'd keep early layers relatively narrow (they only need to learn generic, reusable features like edges) and allow width to grow modestly in deeper layers where features become more task-specific.
- **Efficiency-specific architecture choices:**
  - **Depthwise separable convolutions** (as in MobileNet): factor a standard convolution into a per-channel spatial convolution plus a 1×1 pointwise convolution, cutting parameters and FLOPs by roughly 8–9× for a 3×3 kernel with minimal accuracy loss — well-suited to real-time inspection hardware.
  - **Bottleneck layers**: compress channel dimensions with a 1×1 convolution before an expensive 3×3 operation, then expand back out, reducing compute in the costly spatial convolution.
  - **Global average pooling** instead of large fully connected layers at the head, to avoid the parameter explosion FC layers typically introduce.

**Bottom line:** prioritize an efficient backbone (MobileNet/EfficientNet-style) with moderate depth and channel width, tuned by validation performance, over maximizing either dimension for its own sake.

## 4. Transfer Learning

With 50,000 labeled images — solid but not massive by deep-learning standards — transfer learning is highly valuable rather than optional.

- **Approach:** Start from a model pretrained on a large, general image dataset (ImageNet), ideally using a mobile-efficient backbone (MobileNetV2/V3, EfficientNet-Lite) so the pretrained weights are already aligned with our deployment constraints. Freeze or lightly fine-tune the early layers (they've already learned generic, transferable features — edges, textures, gradients — that apply just as well to metal parts as to natural images), and fine-tune the later layers plus a new classification head on our 50,000-image parts dataset.
- **Why it helps here specifically:**
  - **Faster convergence / less data needed:** The network doesn't have to relearn low-level vision from scratch, so it needs fewer labeled examples and less training time to reach good accuracy — valuable since defect examples are likely a minority class within the 50,000 images.
  - **Reduced overfitting risk:** Reusing well-generalized early features lowers the chance the model memorizes production-line-specific noise instead of learning genuine defect indicators.
  - **Faster time-to-production:** Fine-tuning a pretrained backbone is dramatically cheaper than designing and training a full architecture from scratch, which matters given the business pressure to replace a manual inspection bottleneck quickly.

**Bottom line:** fine-tune a pretrained, mobile-efficient backbone on the parts dataset rather than training from scratch — it directly addresses both the moderate dataset size and the mobile deployment constraint.